# Orbit Wars | OW-Proto

## **Author notes:**  
*"This is the final version of Proto for now. The score peaked around **1080**, which was **Top 95**, stabilized between **1020-1050**, which was around **Top 110-130**. I feel I contributed a good bit to this competition, providing quality code for others to learn and take inspiration from. I will continue competing privately, and likely update this notebook in the future. Good luck to everyone competing and thanks for checking out my work!"* 😄

## Proto scoring formula
```python
score = (100 - dist) + (15 * t.production) + (10 * enemy_bonus) - (0.7 * total_ships) - (2 * eta)
```

Definitions:
- `dist`: distance from home planet to target planet.
- `t`: target planet.
- `enemy_bonus`: extra value when target planet is owned by opponents.
- `total_ships`: total ships needed for capture, including expected production if target is owned.
- `eta`: estimated fleet arrival time.

## Proto-V15 — Final LB Score: Peaked at 1080, stabilized around 1020-1050
Notebook version: **19, 21, 22 (code cleanup)**

**If you want a more detailed version history, check out notebook versions 19 and under.**

**Final main features:**
- Can plan moving planet trajectories to calculate collision angle.
- Dynamic cooperative attacks when one planet cannot capture alone.
- Uses custom optimal target scoring formula.
- Never misses target planets.
- Avoids sending fleets into the sun, also avoids comets completely.
- Sophisticated defense system: calculates at exactly what tick a planet will be vulnerable. Reinforcements are sent and ensured to arrive before the enemy fleets arrive. A planet that's under attack will retain from sending more ships than they can afford, so they won't be left vulnerable.
- Friendly fleet trajectories are kept in mind. Home planets can calculate whether the fleets targeting an enemy planet are insufficient, and send the extra necessary ships.
- Planets are limited to sending max 1 fleet per tick.

**Main issues:**  
- Reinforcement system currently sees all planets as equals, but in reality a planet with production 5 is way more valuable than 1 or 2. Valuable planets are not being prioritized for reinforcements.
- Reinforcements from a planet get comepletely dropped if they won't be able to make it in time before the enemy fleet arrives. Problem with this is that the fleet speed is calculated using the requested ships amount, meaning that the *optimal* amount of resources won't arrive in time, however if planet is valuable (high production), then we could calculate the minimum amount of ships to send in a fleet that will arrive faster than the attacker's fleet, so we could still save the planet.
- Does not account for accidental collisions with other planets in the fleet path.
- A lot more, but that's up to you to fix. 😉

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

## main.py file with agent and logic

In [ ]:
%%writefile main.py
import math
import kaggle_environments.envs.orbit_wars.orbit_wars as ow
import numpy as np

fleet_trajectories = []
reinforcement_trajectories = []
moving_planets = []
planets_coords = {}
steps = 0

MAX_SPEED = 6.0
# could use RL in future to tune these vars to optimal values
MIN_SHIPS_MINE_ATTACK = 5
MIN_SHIPS_TARGET_COOP_ATTACK = 20
COOP_PLANET_CAP = 8
COLLIDE_TICK_THOLD = 1

FORMULA_DIST = 100
FORMULA_PROD_MULT = 15
FORMULA_ENEMY_BONUS_MULT = 10
FORMULA_TOTAL_SHIPS_PERCENT = 0.7


def get_custom_score(m, t):
    dist = math.sqrt((m.x - t.x)**2 + (m.y - t.y)**2)

    min_ships = t.ships + 1
    fleet_speed = get_fleet_speed(max(1, min_ships))
    eta = dist / fleet_speed

    enemy_produced = 0
    enemy_bonus = 0
    if t.owner != -1:
        enemy_produced = eta * t.production
        enemy_bonus = t.production

    total_ships = min_ships + enemy_produced

    # + close targets
    # + high production
    # + if planet is owned by enemy (capturing planet is more valuable because we gain ships, they lose ships)
    # - lot of enemies and enemies produced by arrival
    # - slow arrivals
    
    return (
        (FORMULA_DIST - dist)
        + (FORMULA_PROD_MULT * t.production)
        + (FORMULA_ENEMY_BONUS_MULT * enemy_bonus)
        - (FORMULA_TOTAL_SHIPS_PERCENT * total_ships)
        - (2 * eta)
    )

            
def get_planets_under_attack(mine, fleets, player, vel):
    mov_pl_traj = {}
    under_attack = {}
    seen = set()
    fleets = [f for f in fleets if f.owner != player]
    for m in mine:
        if m.id in moving_planets:
            mov_pl_traj[m.id] = get_planet_trajectories(m, vel)

    for f in fleets:
        fleet_speed = get_fleet_speed(f.ships)
        prev_x = f.x
        prev_y = f.y

        for tick in range(1, 61):
            next_x = f.x + math.cos(f.angle) * fleet_speed * tick
            next_y = f.y + math.sin(f.angle) * fleet_speed * tick

            for m in mine:
                if m.id in moving_planets:
                    m_x, m_y = mov_pl_traj[m.id][tick-1] # tick is 1 based, index 0 based, so -1
                else:
                    m_x, m_y = m.x, m.y
                    
                if collides(prev_x, prev_y, next_x, next_y, m_x, m_y, m.radius): 
                    if (m.id, f.id) not in seen:
                        if m.id not in under_attack:
                            under_attack[m.id] = {
                                "planet": m,
                                "fleets": []
                            }
                            
                        under_attack[m.id]["fleets"].append({
                            "fleet": f,
                            "arrive_tick": tick
                        })
                        seen.add((m.id, f.id))
            
            prev_x = next_x
            prev_y = next_y            
                    
    return under_attack
    


def refresh_local_obs(obs):
    planets = [ow.Planet(*p) for p in obs.get("planets", [])]
    mine = [p for p in planets if p.owner == obs.get("player", [])]
    targets = [p for p in planets if p.owner != obs.get("player", [])]
    player = obs.get("player", -2)
    fleets = [ow.Fleet(*f) for f in obs.get("fleets", [])]

    return {
        "planets": planets,
        "mine": mine,
        "targets": targets,
        "player": player,
        "fleets": fleets
    }


def get_fleet_speed(ships):
    return 1.0 + (MAX_SPEED - 1.0) * (math.log(ships) / math.log(1000)) ** 1.5


def sun_collision(m, fleet_speed, angle, ticks=61):
    prev_x = m.x
    prev_y = m.y

    for tick in range(1, ticks):
        x = m.x + math.cos(angle) * fleet_speed * tick
        y = m.y + math.sin(angle) * fleet_speed * tick

        if collides(prev_x, prev_y, x, y, 50, 50, 10):
            return True

        prev_x = x
        prev_y = y
            
    return False


def calculate_angle(m, t):
    return math.atan2(t.y - m.y, t.x - m.x)
    

def find_angle_to_planet(p, t, ships, vel, moving=False):
    fleet_speed = get_fleet_speed(ships)

    if moving:
        planet_trajectories = get_planet_trajectories(t, vel)
        for tick, (tx, ty) in enumerate(planet_trajectories, start=1):
            dx = tx - p.x
            dy = ty - p.y
            dist_to_target = math.sqrt(dx**2 + dy**2) - p.radius
    
            travel_dist = fleet_speed * tick
            miss_dist = abs(travel_dist - dist_to_target)
    
            if miss_dist > t.radius:
                continue
            
            angle = math.atan2(dy, dx)
            
            if sun_collision(p, fleet_speed, angle):
                return None, None
                
            return angle, tick
        return None, None

    else:
        angle = calculate_angle(p, t)
        
        if sun_collision(p, fleet_speed, angle):
            return None, None

        dist = math.sqrt((p.x - t.x)**2 + (p.y - t.y)**2)
        tick = math.floor(dist / fleet_speed)

        return angle, tick
    return None, None


def collides(x1, y1, x2, y2, cx, cy, r):
    vec_x = x2 - x1
    vec_y = y2 - y1

    vec_to_cx = cx - x1
    vec_to_cy = cy - y1

    vec_length_sq = vec_x**2 + vec_y**2

    if vec_length_sq == 0:
        dx = x1 - cx
        dy = y1 - cy
        return dx**2 + dy**2 <= r**2

    closest_point = (vec_to_cx * vec_x + vec_to_cy * vec_y) / vec_length_sq
    closest_point = max(0, min(1, closest_point))

    closest_x = x1 + closest_point * vec_x
    closest_y = y1 + closest_point * vec_y

    dx = closest_x - cx
    dy = closest_y - cy
    return dx**2 + dy**2 <= r**2


def get_closest_planets_to_target(mine, t):
    planets = []
    for m in mine:
        dist = math.sqrt((m.x - t.x)**2 + (m.y - t.y)**2)
        planets.append((m, dist))
    planets = sorted(planets, key=lambda k: k[1])
    return planets
    

def update_fleet_trajectories(fleets):
    for f_t in fleet_trajectories[:]:
        found = False
        for f in fleets:
            if f.from_planet_id == f_t["mine"].id and abs(f.angle - f_t["angle"]) < 1e-6:
                found = True
                break

        if found:
            f_t["arrive_tick"] = max(0, f_t["arrive_tick"] - 1)

        if not found:
            fleet_trajectories.remove(f_t)


def update_reinforcement_trajectories(planets):
    planet_ids = {p.id for p in planets}
    
    for r_t in reinforcement_trajectories[:]:
        r_t["arrive_tick"] -= 1

        if r_t["arrive_tick"] <= 0:
            reinforcement_trajectories.remove(r_t)
            continue


def get_planet_trajectories(p, vel):
    planet_trajectories = []
    angle = math.atan2(p.y - 50, p.x - 50)
    r = math.sqrt((p.x - 50)**2 + (p.y - 50)**2)
    for tick in range(1, 61): # max 60 ticks
        angle_t = angle + vel * tick
        x_t = 50 + r * math.cos(angle_t)
        y_t = 50 + r * math.sin(angle_t)
        planet_trajectories.append((x_t, y_t))

    return planet_trajectories


def predict_total_ships(m, t, vel, base_ships, m_ships, moving=False):
    total_ships = base_ships
    for _ in range(5):
        angle, arrive_tick = find_angle_to_planet(m, t, total_ships, vel, moving=moving)

        if angle is None:
            return None, None, None
            
        if t.owner != -1:
            new_total_ships = base_ships + arrive_tick * t.production
        else:
            new_total_ships = base_ships

        if new_total_ships > m_ships:
            return None, None, None
        
        if new_total_ships == total_ships:
            break

        total_ships = new_total_ships
    return total_ships, angle, arrive_tick


def plan_coop_attack(attacking_planets, t, base_ships, vel, moving=False):
    remainder = base_ships
    planned = []
    for a_p in attacking_planets:
        p = a_p["planet"]
        p_ships = min(a_p["ships"], remainder)
        
        if p_ships > 0:
            p_ships = min(a_p["ships"], max(p_ships, MIN_SHIPS_MINE_ATTACK))

        if p_ships <= 0:
            continue

        angle, arrive_tick = find_angle_to_planet(p, t, p_ships, vel, moving=moving)

        remainder -= p_ships

        if angle is None or arrive_tick is None:
            continue
            
        planned.append([p, angle, p_ships, arrive_tick])

    return remainder, planned


def get_candidate_targets(m, targets, comet_planet_ids):
    candidate_targets = []
    for t in targets:
        if t.id in comet_planet_ids:
            continue

        score = get_custom_score(m, t)
        candidate_targets.append((m, t, score))

    return sorted(candidate_targets, key=lambda x: x[2], reverse=True)


def fill_moving_planets(obs):
    planets = [ow.Planet(*p) for p in obs.get("planets", [])]
    initial_by_id = {i[0]: ow.Planet(*i) for i in obs.get("initial_planets", [])}
    for p in planets:
        i = initial_by_id[p.id]
        if (p.x, p.y) != (i.x, i.y):
            if p.id not in moving_planets:
                moving_planets.append(p.id)

def get_reinforcement_plans(mine, under_attack):
    reinforcement_plans = {}
    
    for p in mine:
        if p.id in under_attack:
            attacking_fleets = sorted(
                under_attack[p.id]["fleets"],
                key=lambda att: att["arrive_tick"]
            )
            
            incoming_reinforcements = sorted(
                [r for r in reinforcement_trajectories if r["target"].id == p.id],
                key=lambda r: r["arrive_tick"]
            )
            
            p_available_ships = p.ships
            previous_tick = 0
            r_idx = 0

            for att in attacking_fleets:
                att_arrive_tick = att["arrive_tick"]

                p_available_ships += (att_arrive_tick - previous_tick) * p.production
                
                while (
                    r_idx < len(incoming_reinforcements)
                    and incoming_reinforcements[r_idx]["arrive_tick"] <= att_arrive_tick
                ):
                    p_available_ships += incoming_reinforcements[r_idx]["ships"]
                    r_idx += 1

                enemy_ships = att["fleet"].ships
                p_available_ships -= enemy_ships
                previous_tick = att_arrive_tick
                
                if p_available_ships < 0:
                    reinforcements_needed = max(MIN_SHIPS_MINE_ATTACK, abs(p_available_ships))
                    reinforcement_plans[p] = {
                        "ships_needed": reinforcements_needed,
                        "needed_by_tick": att_arrive_tick
                    }
                    break
                
    return reinforcement_plans


def agent(obs):
    global steps
    global fleet_trajectories
    global reinforcement_trajectories
    moves = []
    
    if steps < 2:
        steps += 1
        return []
    if steps == 2:
        fill_moving_planets(obs)
        steps = 3  

    lobs = refresh_local_obs(obs)
    update_fleet_trajectories(lobs.get("fleets", []))
    update_reinforcement_trajectories(lobs.get("planets", []))
    comet_planet_ids = obs.get("comet_planet_ids", [])    
    under_attack = get_planets_under_attack(lobs.get("mine", []), lobs.get("fleets", []), lobs.get("player", -2), obs.angular_velocity)
    exhausted_planets_id = set()
    
    if not lobs.get("targets", []):
        return []
    
    reinforcement_plans = get_reinforcement_plans(lobs.get("mine", []), under_attack)
    for p, plan in reinforcement_plans.items():
        already_reinforced = any(
            r["target"].id == p.id and r["arrive_tick"] >= 0
            for r in reinforcement_trajectories
        )

        if already_reinforced:
            continue
            
        ships_needed = plan["ships_needed"]
        needed_by_tick = plan["needed_by_tick"]
        nearest_planets = get_closest_planets_to_target(lobs.get("mine", []), p)
        
        for row in nearest_planets:
            p_np, _ = row
            
            if p_np.id == p.id or p_np.id in exhausted_planets_id:
                continue

            p_np_available_ships = p_np.ships

            reserved_reinforcement_ships = sum(
                r["ships"]
                for r in reinforcement_trajectories
                if r["mine"].id == p_np.id
            )
            
            p_np_available_ships -= reserved_reinforcement_ships

            if p_np.id in under_attack:
                enemy_ships = sum(
                    att["fleet"].ships
                    for att in under_attack[p_np.id]["fleets"]
                )
                p_np_available_ships = max(0, p_np_available_ships - enemy_ships)

            sent_reinforcements = max(MIN_SHIPS_MINE_ATTACK, ships_needed)

            if p_np_available_ships < sent_reinforcements:
                continue
                
            angle_np = None
            if p.id in moving_planets:
                p_moving = True
                    
            else:
                p_moving = False
                
            angle_np, arrive_tick = find_angle_to_planet(p_np, p, sent_reinforcements, obs.angular_velocity, moving=p_moving)

            if (
                angle_np is None
                or arrive_tick is None
                or arrive_tick > needed_by_tick
            ):
                continue
            
            moves.append([p_np.id, angle_np, sent_reinforcements])
            exhausted_planets_id.add(p_np.id)
            reinforcement_trajectories.append({
                "mine": p_np,
                "target": p,
                "angle": angle_np,
                "ships": sent_reinforcements,
                "arrive_tick": arrive_tick
            })
            break

    for m in sorted(lobs.get("mine", []), key=lambda p: p.ships, reverse=True):
        if m.id in exhausted_planets_id or m.ships < MIN_SHIPS_MINE_ATTACK:
            continue

        candidate_targets = get_candidate_targets(m, lobs.get("targets", []), comet_planet_ids)
    
        for m, t, s in candidate_targets[:3]:
            m_available_ships = m.ships
    
            if m.id in under_attack:
                enemy_ships = sum(
                    att["fleet"].ships
                    for att in under_attack[m.id]["fleets"]
                )
                m_available_ships = max(0, m.ships - enemy_ships)
    
            if m_available_ships < MIN_SHIPS_MINE_ATTACK:
                continue
            
            nearest_planets = get_closest_planets_to_target(lobs.get("mine", []), t)
            safe_nearest_planets = []
            for p, dist in nearest_planets: # check which planets are fit to attack and are not vulnerable
                if p.id == m.id or p.id in exhausted_planets_id:
                    continue
                
                available_ships = p.ships
                
                if p.id in under_attack:
                    enemy_ships = sum(
                        att["fleet"].ships 
                        for att in under_attack[p.id]["fleets"]
                    )
                    available_ships = max(0, p.ships - enemy_ships)
    
                if available_ships < MIN_SHIPS_MINE_ATTACK:
                    continue
                
                safe_nearest_planets.append((p, dist, available_ships))
            
            owned_count = len(lobs.get("mine", []))
            total_count = len(lobs.get("planets", []))
    
            en_route = 0
            if fleet_trajectories:
                en_route = sum(
                    f["ships"]
                    for f in fleet_trajectories
                    if f["target"].id == t.id
                )
    
            needed_now = t.ships + 1
            if t.owner != -1:
                needed_now += 3 * t.production
                
            if owned_count < total_count * 0.75: # release all havoc when targets less than ~25%
                if en_route >= needed_now:
                    continue
            
            base_ships = max(MIN_SHIPS_MINE_ATTACK, needed_now - en_route)
    
            if m_available_ships >= base_ships: # single attack
                if t.id in moving_planets: # single moving planet
                    t_moving = True
                    
                else: # single static planet
                    t_moving = False
                
                total_ships, angle, arrive_tick = predict_total_ships(
                    m, 
                    t, 
                    obs.angular_velocity, 
                    base_ships, 
                    m_available_ships, 
                    moving=t_moving
                )                    
                        
                if angle is not None and arrive_tick is not None:
                    fleet_speed = get_fleet_speed(max(1, total_ships))

                    collides_sun = sun_collision(m, fleet_speed, angle)
                    if collides_sun:
                        continue
                        
                    moves.append([m.id, angle, total_ships])
                    exhausted_planets_id.add(m.id)
                    fleet_trajectories.append({
                        "mine": m,
                        "target": t,
                        "angle": angle,
                        "ships": total_ships,
                        "arrive_tick": arrive_tick
                    })
            
            elif m_available_ships < base_ships and len(lobs.get("mine", [])) > 1 and t.ships >= MIN_SHIPS_TARGET_COOP_ATTACK: # coop attack
                accum = m_available_ships
                attacking_planets = [{"planet": m, "ships": m_available_ships}]
                coop_sent = False
                
                for p, dist, p_available_ships in safe_nearest_planets:
                    if coop_sent:
                        break
                    
                    attacking_planets.append({"planet": p, "ships": p_available_ships})
                    accum += p_available_ships
    
                    if len(attacking_planets) > COOP_PLANET_CAP:
                        break
                    
                    if accum < base_ships:
                        continue
                        
                    if t.id in moving_planets: # coop static owned & unowned
                        t_moving = True
                        
                    else: # coop moving owned & unowned
                        t_moving = False
                        
                    remainder, planned = plan_coop_attack(attacking_planets, t, base_ships, obs.angular_velocity, moving=False)                           

                    if remainder > 0:
                        continue
                        
                    for move in planned:
                        fleet_trajectories.append({
                            "mine": move[0],
                            "target": t,
                            "angle": move[1],
                            "ships": move[2],
                            "arrive_tick": move[3]
                        })
                        exhausted_planets_id.add(move[0].id)
                        move[0] = move[0].id
                        moves.append(move)

                    coop_sent = True
                    break                               
    return moves

## Local testing
Due to crashing when rendering in the notebook, replay is saved in `/kaggle/working` on the right side of your screen as `replay.html`.  
Simply download the file, and open it via your browser of choice.

In [ ]:
# from kaggle_environments import make

# env = make("orbit_wars", debug=True)
# env.run(["main.py", "main.py"])

# final = env.steps[-1]
# for i, s in enumerate(final):
#     print(f"Player {i}: reward={s.reward}, status={s.status}")

# with open("replay.html", "w", encoding="utf-8") as f:
#     f.write(env.render(mode="html"))

## Submission
To make a submission, save (commit) a version of your notebook, then click `Submit Agent` button at top of competition page, then select your latest notebook version and main.py as submission file.